In [ ]:
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
from pathlib import Path
import openslide
import PIL.Image

sys.path.append("/workspaces/TRIDENT/tutorials")

from trident.wsi_objects.entropy_segmentation import segmentation_mask

In [ ]:
from skimage.filters import threshold_otsu
from skimage.transform import resize
from skimage.color import rgb2hsv


def thumbnail_segmentation(
    slide_path: str,
    upscale_factor: int = 4,
):
    fig, axs = plt.subplots(7, 1, figsize=(10, 35))
    entropy_mask = segmentation_mask(slide_path)
    mask_height, mask_width = entropy_mask.shape
    # Calculate the size of the thumbnail
    height = int(mask_height * upscale_factor)
    width = int(mask_width * upscale_factor)

    # cast to uint8
    entropy_mask = (entropy_mask * 255).astype(np.uint8)
    upscaled_mask = resize(
        entropy_mask,
        (height, width),
        anti_aliasing=True,
        order=1,
    )
    axs[0].imshow(upscaled_mask, cmap="gray")
    axs[0].set_title("Upscaled Mask")
    axs[0].axis("off")

    # Load the WSI
    with openslide.OpenSlide(slide_path) as slide:
        thumbnail = slide.get_thumbnail((height, width))

        # Ensure that the thumbnail is actually the correct size
        if thumbnail.size != (width, height):
            thumbnail = thumbnail.resize((width, height), PIL.Image.LANCZOS)
        
        axs[1].imshow(thumbnail)
        axs[1].set_title("Thumbnail")
        axs[1].axis("off")
        
        # Convert to numpy array
        thumbnail_np = np.array(thumbnail)
        
        # Set everything in the thumbnail to white where mask is 0
        thumbnail_np[upscaled_mask == 0] = [255, 255, 255]

        axs[2].imshow(thumbnail_np)
        axs[2].set_title("Masked Thumbnail")
        axs[2].axis("off")

    # Convert to HSV
    thumbnail_hsv = rgb2hsv(thumbnail_np)
    hue_channel = thumbnail_hsv[:, :, 0]
    saturation_channel = thumbnail_hsv[:, :, 1]
    value_channel = thumbnail_hsv[:, :, 2]

    # Threshold 
    hue_threshold = threshold_otsu(hue_channel)
    hue_mask = hue_channel > hue_threshold
    axs[3].imshow(hue_mask, cmap="gray")
    axs[3].set_title("Hue Mask")
    axs[3].axis("off")

    saturation_threshold = threshold_otsu(saturation_channel)
    saturation_mask = saturation_channel > saturation_threshold
    axs[4].imshow(saturation_mask, cmap="gray")
    axs[4].set_title("Saturation Mask")
    axs[4].axis("off")

    value_threshold = threshold_otsu(value_channel)
    value_mask = value_channel < value_threshold
    axs[5].imshow(value_mask, cmap="gray")
    axs[5].set_title("Value Mask")
    axs[5].axis("off")

    # Combine masks
    combined_mask = hue_mask & saturation_mask & value_mask
    axs[6].imshow(combined_mask, cmap="gray")
    axs[6].set_title("Combined Mask")
    axs[6].axis("off")
    fig.tight_layout()
    fig.show()
    return combined_mask


In [ ]:
example_path = Path("/mnt/IMP_CRC/CRS1/slides/CRC_1131.svs")

def get_thumbnail(slide_path: Path, height: int, width: int) -> PIL.Image.Image:
    """
    Get the thumbnail of the slide.
    """
    slide = openslide.OpenSlide(str(slide_path))
    thumbnail = slide.get_thumbnail((width, height))
    slide.close()
    thumbnail = thumbnail.convert("RGB")
    return thumbnail

mask = segmentation_mask(example_path)
mask_height, mask_width = mask.shape


# fig, ax = plt.subplots(2 + len(heatmaps), 1, figsize=(10, (2 + len(heatmaps)) * 6))
thumbnail = get_thumbnail(example_path, mask_height, mask_width)
mask_pil = PIL.Image.fromarray(mask).convert("RGB")
# for i, img  in enumerate([thumbnail, mask_pil] + heatmaps):
#     ax[i].imshow(img)
#     if i == 0:
#         ax[i].set_title("Thumbnail")
#     elif i == 1:
#         ax[i].set_title("Segmentation Mask")
#     else:
#         ax[i].set_title(f"Heatmap {i - 1}")
#     ax[i].axis("off")
# plt.tight_layout()
# plt.show()

thumbnail_segmentation(
    example_path,
    mask,
    upscale_factor=4,
)